# Inverted Index

The text files are placed in some folder.

The **wholeTextFiles** will read the files, and output a tuple (filename, fulltext)

wholeTextFiles -> (filename, fulltext)

In the first step (flatMap), we map our existing key/value pair (filename, fulltext) to a new key-value pair comprising of document-id (i,e, filename) and token (each word in the fulltext) as the key and 1 (representing count) as the value. We will be using the flatMap transformation here to combine all the tokens in a single list.

 flatMap -> ((token, filename), 1)

In the second step (reduceByKey), we will be grouping the key/value pairs with the common key and further aggregate the values for the same key to get the term frequency for a particular word corresponding to its filename.

reduceByKey -> ((token, filename), 1+1+1...)

In the third step, we will change the key/value pairs to a new set of key/ value/pairs with tokens as the key and its filename and respective term frequency as the values.

map -> ((token,count),  filename), notice the count here is also called "Term Frequency"

In [3]:
from pyspark.sql.session import SparkSession
import os, re
import time

spark = SparkSession.builder.master('local').appName('MyInvIndex').getOrCreate()
#spark = SparkSession.builder.master('spark://johnzhang10:7077').appName('MyInvIndex').getOrCreate()
sc = spark.sparkContext
sc._conf.set('spark.executor.memory', '8g')# default is 512M
sc._conf.set('spark.driver.memory', '8g') # default is 512M
sc._conf.set('spark.driver.maxResultsSize', '0')  #default is 1G, 0 means no limit

path = 'E:/spark_dataset/books_500M' #local format
#path = 'hdfs://localhost:9000/hdfs/books500m'  #hdfs format
start = time.time()

#filename without file extension
#inverted_index = sc.wholeTextFiles(path)\
#                .flatMap(lambda x: [((i.lower(),os.path.basename(x[0]).split(".")[0]) ,1) for i in re.split('\W+', x[1])])\
#                .reduceByKey(lambda a, b: a + b)\
#                .map(lambda x: ((x[0][0],x[1]),x[0][1]))\
#                .reduceByKey(lambda a, b: a + b)\
#                .sortByKey()
    
#filename with file extension
inverted_index = sc.wholeTextFiles(path)\
                .flatMap(lambda x: [((i.lower(),os.path.basename(x[0])), 1) for i in re.split('\W+', x[1])])\
                .filter(lambda x: x[0][0] != '')\
                .reduceByKey(lambda a, b: a + b)\
                .map(lambda x: (x[0][0],(x[1],x[0][1])))\
                .reduceByKey(lambda a, b: a + b)
               # .sortByKey()  #add this will fail
    
#save  inverted index to a folder
# make sure there is no following folder yet, otherwise, you need delete it first
inverted_index.saveAsTextFile('./InvIndex_500m')
#inverted_index.saveAsTextFile('hdfs://johnzhang10:9000/hdfs/output/InvIndex_500m')                  
end = time.time()
print(end-start, " seconds")
sc.stop()

831.2084558010101  seconds


## Run step by step

In [4]:
from pyspark.sql.session import SparkSession
import os, re

spark = SparkSession.builder.master('local').appName('MyInvIndex').getOrCreate()
sc = spark.sparkContext

path = 'E:/spark_dataset/books_500M' #local format
#path = 'hdfs://johnzhang10:9000/hdfs/book500m'  #hdfs format

rdd1 = sc.wholeTextFiles(path)\
        .flatMap(lambda x: [((i.lower(),os.path.basename(x[0])), 1) for i in re.split('\W+', x[1])])\
        .filter(lambda x: x[0][0] != '')
    
rdd1.take(10)

[(('the', '0.txt'), 1),
 (('project', '0.txt'), 1),
 (('gutenberg', '0.txt'), 1),
 (('ebook', '0.txt'), 1),
 (('of', '0.txt'), 1),
 (('frankenstein', '0.txt'), 1),
 (('by', '0.txt'), 1),
 (('mary', '0.txt'), 1),
 (('wollstonecraft', '0.txt'), 1),
 (('godwin', '0.txt'), 1)]

In [5]:
rdd2 = rdd1.reduceByKey(lambda a, b: a+b)
rdd2.take(10)

[(('online', '0.txt'), 4),
 (('language', '0.txt'), 20),
 (('petersburgh', '0.txt'), 3),
 (('dear', '0.txt'), 72),
 (('question', '0.txt'), 8),
 (('make', '0.txt'), 39),
 (('aroused', '0.txt'), 1),
 (('reverence', '0.txt'), 4),
 (('strove', '0.txt'), 3),
 (('permission', '0.txt'), 10)]

In [12]:
invIndex = rdd2.map(lambda x: (x[0][0],(x[1],x[0][1])))\
                    .reduceByKey(lambda a, b: a + b)
                    
invIndex.take(10)

[('abject',
  (2,
   '108.txt',
   1,
   '497.txt',
   1,
   '152.txt',
   4,
   '18.txt',
   2,
   '737.txt',
   1,
   '456.txt',
   1,
   '508.txt',
   3,
   '666.txt',
   1,
   '216.txt',
   1,
   '651.txt',
   1,
   '11.txt',
   1,
   '328.txt',
   1,
   '436.txt',
   1,
   '530.txt',
   2,
   '846.txt',
   2,
   '101.txt',
   4,
   '278.txt',
   3,
   '31.txt',
   1,
   '416.txt',
   1,
   '779.txt',
   2,
   '97.txt',
   1,
   '756.txt',
   1,
   '170.txt',
   1,
   '921.txt',
   15,
   '199.txt',
   1,
   '856.txt',
   1,
   '904.txt',
   6,
   '241.txt',
   1,
   '256.txt',
   1,
   '376.txt',
   1,
   '740.txt',
   1,
   '976.txt',
   1,
   '361.txt',
   1,
   '795.txt',
   2,
   '925.txt',
   7,
   '831.txt',
   1,
   '61.txt',
   2,
   '253.txt',
   1,
   '827.txt',
   1,
   '558.txt',
   1,
   '731.txt',
   1,
   '850.txt',
   8,
   '404.txt',
   1,
   '455.txt',
   1,
   '356.txt',
   1,
   '301.txt',
   2,
   '501.txt',
   1,
   '901.txt',
   1,
   '745.txt',
   1,
   '38

In [13]:
#save inverted index to a folder
# make sure there is no the following folder yet, otherwise, you need delete it first
invIndex.saveAsTextFile('./InvIndex2_500m')
sc.stop()